In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv("C:\\Users\\julia\\Prosjekter\\ndms\\data\\forbruk_per_nettomraade.csv", sep=';', decimal=',')

In [2]:
df['FORBRUK (I MWH)'] = df['VOLUM_KWH'] / 1000
df['TID'] = pd.to_datetime(df['STARTTID'], utc=True).dt.tz_convert('Europe/Oslo')
df['FORBRUKSGRUPPE'] = df['FORBRUKSGRUPPE'].astype('category')
df['FORBRUK (I MWH)'] = pd.to_numeric(df['FORBRUK (I MWH)'], errors='coerce').astype('float64')

In [3]:
df.drop(columns=['VOLUM_KWH', 'STARTTID','SLUTTID'])

,NETTOMRÅDE,FORBRUKSGRUPPE,ANTALL_MÅLEPUNKT,FORBRUK (I MWH),TID
0,AEN2 DN,Hytter og fritidseiendommer,29278,26.454936,2022-02-01 00:00:00+01:00
1,AEN2 DN,Husholdning,141769,270.146644,2022-02-01 00:00:00+01:00
2,AEN2 DN,Primær Industri,1971,6.190081,2022-02-01 00:00:00+01:00
3,AEN2 DN,Sekundær Industri,5699,51.180987,2022-02-01 00:00:00+01:00
4,AEN2 DN,Tertiære Tjenester,26092,155.623755,2022-02-01 00:00:00+01:00
...,...,...,...,...,...
9664310,ØEIKER1,Hytter og fritidseiendommer,159,0.062942,2026-04-30 23:00:00+02:00
9664311,ØEIKER1,Husholdning,9899,14.398304,2026-04-30 23:00:00+02:00
9664312,ØEIKER1,Primær Industri,169,0.424127,2026-04-30 23:00:00+02:00
9664313,ØEIKER1,Sekundær Industri,193,2.454670,2026-04-30 23:00:00+02:00


In [4]:
glitre_vinter = df.loc[
    (df['NETTOMRÅDE'] == 'GLITRE D1') & 
    (df['TID'] >= '2026-01-28') & 
    (df['TID'] < '2026-01-29'),
    ['TID','FORBRUKSGRUPPE', 'FORBRUK (I MWH)']
]
glitre_vinter.reset_index(inplace=True, drop=True)

In [5]:
# Morgen: 06-12, ettermiddag: 12-16, kveld: 16-21, natt: 21-06
from IPython.display import HTML, display

result = glitre_vinter.pivot(
    index='TID', columns='FORBRUKSGRUPPE', values='FORBRUK (I MWH)'
).sort_index()
result.columns.name = None
result['TOTAL_FORBRUK'] = result.sum(axis=1)

# Lag en egen visning slik at result beholder tallene og tidsindeksen.
tabell = result.rename(columns={'TOTAL_FORBRUK': 'Totalt'}).copy()
time = tabell.index.hour
tabell.insert(0, 'Periode', np.select(
    [(time >= 6) & (time < 12),
     (time >= 12) & (time < 16),
     (time >= 16) & (time < 21)],
    ['Morgen', 'Ettermiddag', 'Kveld'],
    default='Natt',
))
tabell.index = tabell.index.strftime('%d.%m.%Y %H:%M')
tabell.index.name = 'Tid'

# to_html trenger ikke Jinja2. Formatering gjelder bare visningen.
html = tabell.to_html(
    table_id='forbruk-tabell',
    float_format=lambda verdi: f'{verdi:,.2f}'.replace(',', ' ').replace('.', ','),
    na_rep='-',
    border=0,
)
display(HTML("""
<style>
#forbruk-tabell { border-collapse: collapse; }
#forbruk-tabell th, #forbruk-tabell td { padding: 6px 12px; text-align: right; }
#forbruk-tabell td:first-of-type { text-align: left; }
#forbruk-tabell td:last-child { font-weight: bold; }
</style>
<p><strong>Forbruk per time (MWh) - lokal tid i Oslo</strong></p>
""" + html))


,Periode,Husholdning,Hytter og fritidseiendommer,Primær Industri,Sekundær Industri,Tertiære Tjenester,Totalt
Tid,,,,,,,
28.01.2026 00:00,Natt,"212,57","3,13","13,15","43,72","82,81","355,38"
28.01.2026 01:00,Natt,"205,75","3,15","13,56","43,98","82,01","348,46"
28.01.2026 02:00,Natt,"200,61","3,16","14,89","44,53","81,33","344,53"
28.01.2026 03:00,Natt,"197,18","3,16","15,10","44,62","82,38","342,44"
28.01.2026 04:00,Natt,"195,41","3,17","15,45","45,94","83,26","343,23"
28.01.2026 05:00,Natt,"197,09","3,21","15,71","48,25","89,17","353,43"
28.01.2026 06:00,Morgen,"206,42","3,24","15,74","54,11","102,60","382,11"
28.01.2026 07:00,Morgen,"217,11","3,27","15,73","62,38","119,34","417,82"
28.01.2026 08:00,Morgen,"213,07","3,25","16,21","65,91","132,38","430,81"


In [7]:
glitre_vinter = df.loc[
    (df['NETTOMRÅDE'] == 'GLITRE D1') & 
    (df['TID'] >= '2026-01-29') & 
    (df['TID'] < '2026-01-30'),
    ['TID','FORBRUKSGRUPPE', 'FORBRUK (I MWH)']
]
glitre_vinter.reset_index(inplace=True, drop=True)

# Morgen: 06-12, ettermiddag: 12-16, kveld: 16-21, natt: 21-06
from IPython.display import HTML, display

result = glitre_vinter.pivot(
    index='TID', columns='FORBRUKSGRUPPE', values='FORBRUK (I MWH)'
).sort_index()
result.columns.name = None
result['TOTAL_FORBRUK'] = result.sum(axis=1)

# Lag en egen visning slik at result beholder tallene og tidsindeksen.
tabell = result.rename(columns={'TOTAL_FORBRUK': 'Totalt'}).copy()
time = tabell.index.hour
tabell.insert(0, 'Periode', np.select(
    [(time >= 6) & (time < 12),
     (time >= 12) & (time < 16),
     (time >= 16) & (time < 21)],
    ['Morgen', 'Ettermiddag', 'Kveld'],
    default='Natt',
))
tabell.index = tabell.index.strftime('%d.%m.%Y %H:%M')
tabell.index.name = 'Tid'

# to_html trenger ikke Jinja2. Formatering gjelder bare visningen.
html = tabell.to_html(
    table_id='forbruk-tabell',
    float_format=lambda verdi: f'{verdi:,.2f}'.replace(',', ' ').replace('.', ','),
    na_rep='-',
    border=0,
)
display(HTML("""
<style>
#forbruk-tabell { border-collapse: collapse; }
#forbruk-tabell th, #forbruk-tabell td { padding: 6px 12px; text-align: right; }
#forbruk-tabell td:first-of-type { text-align: left; }
#forbruk-tabell td:last-child { font-weight: bold; }
</style>
<p><strong>Forbruk per time (MWh) - lokal tid i Oslo</strong></p>
""" + html))

,Periode,Husholdning,Hytter og fritidseiendommer,Primær Industri,Sekundær Industri,Tertiære Tjenester,Totalt
Tid,,,,,,,
29.01.2026 00:00,Natt,"219,93","3,33","13,07","45,92","84,90","367,13"
29.01.2026 01:00,Natt,"213,05","3,33","13,65","45,96","84,60","360,59"
29.01.2026 02:00,Natt,"208,77","3,34","14,58","45,70","83,85","356,24"
29.01.2026 03:00,Natt,"205,56","3,32","14,34","46,08","83,95","353,26"
29.01.2026 04:00,Natt,"203,41","3,31","16,08","47,33","84,89","355,03"
29.01.2026 05:00,Natt,"204,99","3,30","17,18","49,12","90,69","365,28"
29.01.2026 06:00,Morgen,"215,04","3,29","16,41","55,70","105,46","395,91"
29.01.2026 07:00,Morgen,"225,48","3,35","17,47","64,49","122,94","433,74"
29.01.2026 08:00,Morgen,"223,94","3,37","18,47","68,64","135,85","450,27"
